In [239]:
# !pip install pandas

In [240]:
#!pip install scikit-learn

In [241]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [242]:
df_main = pd.read_csv("vgsales.csv")
df_main['Name_lower'] = df_main['Name'].str.lower().str.strip()
df_main.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Name_lower
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,wii sports
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,super mario bros.
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,mario kart wii
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,wii sports resort
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,pokemon red/pokemon blue


In [243]:
steam_df = pd.read_csv("steam_games.csv", encoding='utf-8', low_memory=False)
steam_df['name_lower'] = steam_df['name'].str.lower().str.strip()
steam_df.head()

,url,types,name,desc_snippet,recent_reviews,all_reviews,release_date,developer,publisher,popular_tags,...,languages,achievements,genre,game_description,mature_content,minimum_requirements,recommended_requirements,original_price,discount_price,name_lower
0,https://store.steampowered.com/app/379720/DOOM/,app,DOOM,Now includes all three premium DLC packs (Unto...,"Very Positive,(554),- 89% of the 554 user revi...","Very Positive,(42,550),- 92% of the 42,550 use...","May 12, 2016",id Software,"Bethesda Softworks,Bethesda Softworks","FPS,Gore,Action,Demons,Shooter,First-Person,Gr...",...,"English,French,Italian,German,Spanish - Spain,...",54.0,Action,"About This Game Developed by id software, the...",NaN,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers...",$19.99,$14.99,doom
1,https://store.steampowered.com/app/578080/PLAY...,app,PLAYERUNKNOWN'S BATTLEGROUNDS,PLAYERUNKNOWN'S BATTLEGROUNDS is a battle roya...,"Mixed,(6,214),- 49% of the 6,214 user reviews ...","Mixed,(836,608),- 49% of the 836,608 user revi...","Dec 21, 2017",PUBG Corporation,"PUBG Corporation,PUBG Corporation","Survival,Shooter,Multiplayer,Battle Royale,PvP...",...,"English,Korean,Simplified Chinese,French,Germa...",37.0,"Action,Adventure,Massively Multiplayer",About This Game PLAYERUNKNOWN'S BATTLEGROUND...,Mature Content Description The developers de...,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",$29.99,NaN,playerunknown's battlegrounds
2,https://store.steampowered.com/app/637090/BATT...,app,BATTLETECH,Take command of your own mercenary outfit of '...,"Mixed,(166),- 54% of the 166 user reviews in t...","Mostly Positive,(7,030),- 71% of the 7,030 use...","Apr 24, 2018",Harebrained Schemes,"Paradox Interactive,Paradox Interactive","Mechs,Strategy,Turn-Based,Turn-Based Tactics,S...",...,"English,French,German,Russian",128.0,"Action,Adventure,Strategy",About This Game From original BATTLETECH/Mec...,NaN,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",$39.99,NaN,battletech
3,https://store.steampowered.com/app/221100/DayZ/,app,DayZ,The post-soviet country of Chernarus is struck...,"Mixed,(932),- 57% of the 932 user reviews in t...","Mixed,(167,115),- 61% of the 167,115 user revi...","Dec 13, 2018",Bohemia Interactive,"Bohemia Interactive,Bohemia Interactive","Survival,Zombies,Open World,Multiplayer,PvP,Ma...",...,"English,French,Italian,German,Spanish - Spain,...",NaN,"Action,Adventure,Massively Multiplayer",About This Game The post-soviet country of Ch...,NaN,"Minimum:,OS:,Windows 7/8.1 64-bit,Processor:,I...","Recommended:,OS:,Windows 10 64-bit,Processor:,...",$44.99,NaN,dayz
4,https://store.steampowered.com/app/8500/EVE_On...,app,EVE Online,EVE Online is a community-driven spaceship MMO...,"Mixed,(287),- 54% of the 287 user reviews in t...","Mostly Positive,(11,481),- 74% of the 11,481 u...","May 6, 2003",CCP,"CCP,CCP","Space,Massively Multiplayer,Sci-fi,Sandbox,MMO...",...,"English,German,Russian,French",NaN,"Action,Free to Play,Massively Multiplayer,RPG,...",About This Game,NaN,"Minimum:,OS:,Windows 7,Processor:,Intel Dual C...","Recommended:,OS:,Windows 10,Processor:,Intel i...",Free,NaN,eve online


In [244]:
merged = pd.merge(
    df_main,
    steam_df,
    left_on='Name_lower',
    right_on='name_lower',
    how='inner'
)

In [245]:
merged_subset = merged[
    [
        'Name',           
        'Platform',
        'Year',
        'Genre',
        'Publisher',       
        'popular_tags',     
        'desc_snippet',    
    ]
].copy()

merged_subset.columns = [
    'Name', 'Platform', 'Year', 'Genre',
    'Publisher', 'Tags', 'DescriptionShort'
]

In [246]:
steam_only = steam_df[~steam_df['name_lower'].isin(df_main['Name_lower'])]

steam_only_subset = steam_only[
    ['name', 'popular_tags', 'developer', 'desc_snippet', 'release_date']
].copy()

steam_only_subset.columns = [
    'Name', 'Tags', 'Publisher', 'DescriptionShort', 'Year'
]

steam_only_subset['Platform'] = ''
steam_only_subset['Genre'] = ''

In [247]:
df_combined = pd.concat([merged_subset, steam_only_subset], ignore_index=True)
df_combined.head()

,Name,Platform,Year,Genre,Publisher,Tags,DescriptionShort
0,Grand Theft Auto V,PS3,2013.0,Action,Take-Two Interactive,"Open World,Action,Multiplayer,Third Person,Fir...","Los Santos is a city of bright lights, long ni..."
1,Grand Theft Auto: San Andreas,PS2,2004.0,Action,Take-Two Interactive,"Open World,Action,Crime,Classic,Third Person,S...",Five years ago Carl Johnson escaped from the p...
2,Grand Theft Auto V,X360,2013.0,Action,Take-Two Interactive,"Open World,Action,Multiplayer,Third Person,Fir...","Los Santos is a city of bright lights, long ni..."
3,Grand Theft Auto: Vice City,PS2,2002.0,Action,Take-Two Interactive,"Open World,Action,1980s,Classic,Great Soundtra...",Welcome to Vice City. Welcome to the 1980s. Fr...
4,Call of Duty: Black Ops,X360,2010.0,Shooter,Activision,"Action,FPS,Zombies,Multiplayer,Shooter,Singlep...",The biggest first-person action series of all ...


In [248]:
df_combined['Tags'] = (
    df_combined['Tags']
    .astype(str)
    .str.replace(',', ' ')
    .str.replace('[^a-zA-Z0-9 ]', '', regex=True)
    .str.lower()
)

df_combined['DescriptionShort'] = df_combined['DescriptionShort'].fillna('')
df_combined['Publisher'] = df_combined['Publisher'].fillna('')
df_combined['Genre'] = df_combined['Genre'].fillna('')
df_combined['Platform'] = df_combined['Platform'].fillna('')

In [249]:
df_combined['Feature_Text'] = (
    df_combined['Genre'].fillna("") + " " +
    df_combined['Tags'].fillna("") + " " +
    df_combined['Publisher'].fillna("") + " " +
    df_combined['DescriptionShort'].fillna("")
)

In [250]:
if os.path.exists("user_games.csv"):
    df_memory = pd.read_csv("user_games.csv")
else:
    df_memory = pd.DataFrame(columns=["Name", "Platform", "Genre", "Publisher", "Tags", "DescriptionShort"])

In [251]:
from sklearn.metrics.pairwise import cosine_similarity


vectorizer = HashingVectorizer(
    n_features=2**14,
    stop_words='english',
    alternate_sign=False
)

feature_matrix = vectorizer.transform(df_combined['Feature_Text'])

In [252]:
def merge_platforms(df):
    grouped = (
        df.groupby("Name")
          .agg({
              "Genre": "first",
              "Publisher": "first",
              "Platform": lambda x: ", ".join(sorted(set([p for p in x if p]))),
          })
          .reset_index()
    )
    return grouped

In [253]:
def recommend_game(title, top_n=5):
    global df_combined, df_memory, feature_matrix

    title_lower = title.lower().strip()

    # ensure lowercase column exists
    df_combined['name_lower'] = df_combined['Name'].str.lower().str.strip()

    # check if the game exists
    match = df_combined[df_combined['name_lower'] == title_lower]

    # add games if not in dataset
    if match.empty:
        print("❌ Game not found in dataset.")
        print("Let's add it! Please provide the following details:\n")

        platform = input("Platform: ")
        genre = input("Genre: ")
        publisher = input("Publisher: ")
        tags = input("Tags (comma-separated): ")
        description = input("Short description: ")

        new_row = {
            "Name": title,
            "Platform": platform,
            "Genre": genre,
            "Publisher": publisher,
            "Tags": tags,
            "DescriptionShort": description
        }

        # save to user memory
        df_memory = pd.concat([df_memory, pd.DataFrame([new_row])], ignore_index=True)
        df_memory.to_csv("user_games.csv", index=False)

        # add to main combined dataset
        df_combined = pd.concat([df_combined, pd.DataFrame([new_row])], ignore_index=True)
        df_combined['name_lower'] = df_combined['Name'].str.lower().str.strip()
        
        # rebuild feature text + matrix
        df_combined['Feature_Text'] = (
            df_combined['Genre'].fillna("") + " " +
            df_combined['Tags'].fillna("") + " " +
            df_combined['Publisher'].fillna("") + " " +
            df_combined['DescriptionShort'].fillna("")
        )
        feature_matrix = vectorizer.transform(df_combined['Feature_Text'])

        print(f"\n✅ '{title}' added successfully!")
        match = df_combined[df_combined['name_lower'] == title_lower]


    # find similarity for already exisitng games in dataset
    idx = match.index[0]
    game_vector = feature_matrix[idx]
    similarities = cosine_similarity(game_vector, feature_matrix)[0]

    # sort by similarity
    ranked = similarities.argsort()[::-1]

    # exclude itself & prequel/sequel games
    base_name = title.split(":")[0].split(" ")[0].lower()

    results = []
    seen_names = set()

    for i in ranked:
        if i == idx:
            continue

        row = df_combined.iloc[i]
        game_name = row['Name']

        if base_name in game_name.lower():
            continue


        if game_name in seen_names:
            continue

        seen_names.add(game_name)
        results.append(i)

        if len(results) == top_n:
            break

    print(f"\n🎮 Games similar to '{title}':\n")

    # formatting for printing
    for i in results:
        row = df_combined.iloc[i]

        duplicates = df_combined[df_combined['Name'] == row['Name']]
        merged = merge_platforms(duplicates).iloc[0]

        fields = []
        if isinstance(merged['Genre'], str) and merged['Genre'].strip():
            fields.append(merged['Genre'])
        if isinstance(merged['Platform'], str) and merged['Platform'].strip():
            fields.append(merged['Platform'])
        if isinstance(merged['Publisher'], str) and merged['Publisher'].strip():
            fields.append(merged['Publisher'])

        fields_text = " | ".join(fields)

        print(f"- {merged['Name']}" + (f" | {fields_text}" if fields_text else ""))

In [254]:
user_input = input("🎮 Enter a game title: ")


In [255]:
recommend_game(user_input)

❌ Game not found in dataset.
Let's add it! Please provide the following details:


✅ 'Valorant' added successfully!

🎮 Games similar to 'Valorant':

- Stories In Stone | 16 Bit Psych,Kyle B
- Mega Man X5 Sound Collection | CAPCOM CO., LTD
- Fantasy Grounds - Quests of Doom 4: A Midnight Council of Quail (5E) | SmiteWorks USA, LLC
- Rocksmith® 2014 Edition – Remastered – Stone Temple Pilots - “Trippin’ on a Hole in a Paper Heart” | Ubisoft - San Francisco
- Rocksmith® 2014 Edition – Remastered – Sabaton - “Ghost Division” | Ubisoft - San Francisco
